# Explore Simrad .raw File Format

Investigate the binary structure of .raw files to understand:
- Datagram types and structure
- Metadata/configuration sections
- Range and depth parameters

In [ ]:
import sys
sys.path.insert(0, '../..')

from pathlib import Path
import numpy as np
from raw_sensor_integration import raw_parser, metadata

In [ ]:
# Path to raw files
RAW_DIR = Path('/Users/axelhagen/Downloads/drive-download-20251116T003920Z-1-001')
raw_files = list(RAW_DIR.glob('*.raw'))
print(f"Found {len(raw_files)} raw files:")
for f in raw_files:
    print(f"  {f.name}")

## Basic File Info

In [ ]:
# Load first file for exploration
sample_file = raw_files[0]
data = raw_parser.read_raw_file(sample_file)

print(f"File: {sample_file.name}")
print(f"Size: {len(data) / 1024 / 1024:.2f} MB")
print(f"\nFirst 200 bytes (hex):")
print(data[:200].hex(' '))

In [ ]:
# Look at first bytes as ASCII
print("First 500 bytes as ASCII (non-printable as dots):")
ascii_repr = ''.join(chr(b) if 32 <= b <= 126 else '.' for b in data[:500])
# Print in lines of 80
for i in range(0, len(ascii_repr), 80):
    print(ascii_repr[i:i+80])

## Find Datagram Types

In [ ]:
# Search for known Simrad datagram types
datagram_types = raw_parser.find_datagram_types(data)

print("Found datagram types:")
for dtype, offsets in sorted(datagram_types.items()):
    print(f"  {dtype}: {len(offsets)} occurrences")
    if len(offsets) <= 5:
        print(f"    at offsets: {offsets}")
    else:
        print(f"    first 5 at: {offsets[:5]}")

In [ ]:
# Look for any 4-letter patterns that repeat often (potential datagram types)
from collections import Counter

# Find all 4-byte sequences that look like type identifiers
potential_types = Counter()
for i in range(len(data) - 4):
    chunk = data[i:i+4]
    # Check if it looks like a type identifier (all printable ASCII)
    if all(32 <= b <= 126 for b in chunk):
        text = chunk.decode('ascii')
        # Common pattern: ends with digit
        if text[-1].isdigit() and text[:-1].isalpha():
            potential_types[text] += 1

print("Most common type-like patterns:")
for pattern, count in potential_types.most_common(20):
    print(f"  {pattern}: {count}")

## Examine Datagram Structure

In [ ]:
# Look at the structure around the first occurrence of each datagram type
for dtype, offsets in sorted(datagram_types.items()):
    offset = offsets[0]
    print(f"\n{'='*60}")
    print(f"Datagram: {dtype} at offset {offset}")
    print('='*60)
    
    # Show bytes before (might contain length)
    start = max(0, offset - 8)
    end = min(len(data), offset + 100)
    
    print(f"\nBytes around datagram [{start}:{end}]:")
    chunk = data[start:end]
    
    # Hex dump with ASCII
    for i in range(0, len(chunk), 16):
        hex_part = chunk[i:i+16].hex(' ')
        ascii_part = ''.join(chr(b) if 32 <= b <= 126 else '.' for b in chunk[i:i+16])
        print(f"{start+i:6d}: {hex_part:<48} {ascii_part}")

## Extract XML Configuration

In [ ]:
# Try to extract XML sections
xml_sections = raw_parser.extract_xml_content(data)

print(f"Found {len(xml_sections)} XML sections")
for i, xml in enumerate(xml_sections):
    print(f"\nSection {i} ({len(xml)} chars):")
    # Print first 2000 chars
    print(xml[:2000])
    if len(xml) > 2000:
        print(f"\n... [{len(xml) - 2000} more chars]")

## Search for Range/Depth Parameters

In [ ]:
# Find text strings that might contain parameter names
strings = raw_parser.find_text_strings(data, min_length=6)

# Filter for interesting keywords
keywords = ['range', 'depth', 'sample', 'frequency', 'pulse', 'power', 
            'gain', 'interval', 'channel', 'transducer']

print("Strings containing range/depth related keywords:")
for offset, string in strings:
    lower = string.lower()
    for kw in keywords:
        if kw in lower:
            print(f"  {offset:8d}: {string[:100]}")
            break

In [ ]:
# Use metadata extractor
meta = metadata.extract_metadata(sample_file)

print("Extracted range-related parameters:")
for key, value in meta['range_info'].items():
    print(f"  {key}: {value}")

## Examine RAW Data Datagrams

These contain the actual ping data.

In [ ]:
# Find RAW datagrams (RAW0, RAW3, etc.)
raw_datagrams = []
for dtype in ['RAW0', 'RAW3']:
    if dtype in datagram_types:
        raw_datagrams.extend([(dtype, off) for off in datagram_types[dtype]])

raw_datagrams.sort(key=lambda x: x[1])
print(f"Found {len(raw_datagrams)} RAW datagrams")

if raw_datagrams:
    # Analyze spacing between datagrams
    if len(raw_datagrams) > 1:
        spacings = [raw_datagrams[i+1][1] - raw_datagrams[i][1] 
                   for i in range(len(raw_datagrams)-1)]
        print(f"Spacing between RAW datagrams:")
        print(f"  Min: {min(spacings)}")
        print(f"  Max: {max(spacings)}")
        print(f"  Mean: {np.mean(spacings):.0f}")
        print(f"  Unique values: {len(set(spacings))}")

In [ ]:
# Examine structure of first RAW datagram
if raw_datagrams:
    dtype, offset = raw_datagrams[0]
    
    # Get length from 4 bytes before
    length = int.from_bytes(data[offset-4:offset], 'little')
    
    print(f"First {dtype} datagram:")
    print(f"  Offset: {offset}")
    print(f"  Length: {length} bytes")
    
    # Show header structure
    header = data[offset:offset+100]
    print(f"\nHeader bytes:")
    for i in range(0, min(100, len(header)), 16):
        hex_part = header[i:i+16].hex(' ')
        ascii_part = ''.join(chr(b) if 32 <= b <= 126 else '.' for b in header[i:i+16])
        print(f"{i:6d}: {hex_part:<48} {ascii_part}")

## Compare Multiple Files

Check if structure is consistent across files.

In [ ]:
# Get basic info for all files
print(f"{'File':<30} {'Size (MB)':<12} {'Datagram Types'}")
print('='*80)

for f in raw_files:
    info = raw_parser.get_file_info(f)
    types = ', '.join(f"{k}({len(v)})" for k, v in info['datagram_types'].items())
    print(f"{info['filename']:<30} {info['size_mb']:<12.2f} {types}")

## Observations

The XML configuration section holds range/depth parameters per channel. RAW3 datagrams
contain the actual ping data, with channel IDs embedded in the header. The two channels
correspond to the 38 kHz and 200 kHz transducers — see notebook 02 for ping-level inspection.

In [ ]:
# Cross-check: count RAW3 datagrams per channel across all files
print("RAW3 ping counts per channel per file:\n")
print(f"{'File':<30} {'38 kHz pings':>14} {'200 kHz pings':>15}")
print('-' * 62)

for f in raw_files:
    cs_list = metadata.get_channel_settings(f)
    # Build a quick ping-count lookup from datagram offsets
    f_data = raw_parser.read_raw_file(f)
    f_dtypes = raw_parser.find_datagram_types(f_data)
    ch_counts = {}
    for off in f_dtypes.get('RAW3', []):
        cid = f_data[off+12:off+140].split(b'\x00')[0].decode('ascii', errors='ignore')
        ch_counts[cid] = ch_counts.get(cid, 0) + 1

    # Map channel → frequency label
    freq_counts = {}
    for cs in cs_list:
        freq = cs.get('frequency')
        label = f"{freq//1000} kHz" if freq else cs['channel_id'][-8:]
        freq_counts[label] = ch_counts.get(cs['channel_id'], 0)

    counts_38  = freq_counts.get('38 kHz', '?')
    counts_200 = freq_counts.get('200 kHz', '?')
    print(f"{f.name:<30} {str(counts_38):>14} {str(counts_200):>15}")

In [ ]:
# Use get_channel_settings to show full per-channel metadata including frequency
channel_settings = metadata.get_channel_settings(sample_file)

print(f"Channel settings for {sample_file.name}:\n")
for cs in channel_settings:
    freq = cs.get('frequency')
    freq_str = f"{freq/1000:.0f} kHz" if freq else "unknown"
    max_range = cs.get('max_range')
    range_str = f"{max_range:.1f} m" if max_range else "N/A"
    print(f"  Channel: {cs['channel_id']}")
    print(f"    Frequency:       {freq_str}")
    print(f"    Sample interval: {cs.get('sample_interval', 'N/A')} s")
    print(f"    Sound velocity:  {cs.get('sound_velocity', 'N/A')} m/s")
    print(f"    Sample count:    {cs.get('sample_count', 'N/A')}")
    print(f"    Max range:       {range_str}")
    print()

In [ ]:
import re

# Search the raw bytes for Frequency= attributes near each channel ID
ch_ids = [
    'WBT 737172-7 ES38-18|200-18CR_ES',
    'WBT 737172-8 ES38-18|200-18CR_ES',
]

freq_pattern = re.compile(rb'Frequency="([^"]+)"')

print(f"Frequency extraction from XML config in: {sample_file.name}\n")

for ch_id in ch_ids:
    ch_bytes = ch_id.encode('ascii')
    pos = data.find(ch_bytes)
    if pos == -1:
        print(f"{ch_id}: NOT FOUND in file")
        continue

    # Search in a 3 kB window around the channel reference
    window = data[max(0, pos - 500): pos + 2500]
    matches = freq_pattern.findall(window)
    freqs_hz = [float(m) for m in matches if float(m) > 0]
    freqs_khz = sorted(set(f / 1000 for f in freqs_hz))
    print(f"Channel: {ch_id}")
    print(f"  Frequencies found (kHz): {freqs_khz}")
    print()

## Frequency Analysis

Identify which channels correspond to which transmit frequencies (e.g. 38 kHz vs 200 kHz).

In [ ]:
import matplotlib.pyplot as plt
from raw_sensor_integration import ping_parser

# Extract pings for both channels (use sorted file list for consistency)
sample_file = sorted(RAW_DIR.glob('*.raw'))[4]  # D20240131-T154447.raw
print(f"Loading pings from: {sample_file.name}")

pings = ping_parser.extract_pings(sample_file, max_pings=2000)
print(f"  Extracted {len(pings)} pings")

# Channel settings tell us which ID is which frequency
cs_list = metadata.get_channel_settings(sample_file)
ch_by_freq = {}
for cs in cs_list:
    freq = cs.get('frequency')
    if freq:
        label = f"{freq // 1000} kHz"
        ch_by_freq[label] = cs['channel_id']

print("\nChannel → frequency mapping:")
for label, ch_id in ch_by_freq.items():
    print(f"  {label}: {ch_id}")

## Echogram Visualizations: 38 kHz vs 200 kHz

Plot both frequency channels side by side. The 38 kHz channel has deeper range (~286 m) while the 200 kHz channel covers a shallower window (~23 m).

In [ ]:
# Build echograms for 38 kHz and 200 kHz channels side by side
power_images = {}
range_axes = {}

for label, ch_id in ch_by_freq.items():
    cs = next(cs for cs in cs_list if cs['channel_id'] == ch_id)
    img = ping_parser.pings_to_power_image(pings, ch_id)
    img = np.clip(img, -100, 0)
    power_images[label] = img

    # Build a depth axis in metres
    n_samples = img.shape[0]
    si = cs['sample_interval']
    sv = cs['sound_velocity']
    range_m = np.arange(n_samples) * si * sv / 2
    range_axes[label] = range_m

    vmin_ch, vmax_ch = np.percentile(img, [2, 98])
    print(f"{label}: {img.shape[0]} depth samples × {img.shape[1]} pings  |  "
          f"range 0–{range_m[-1]:.1f} m  |  dB: {vmin_ch:.0f} … {vmax_ch:.0f}")

# ── Side-by-side echogram plot ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, (label, img) in zip(axes, power_images.items()):
    vmin_ch, vmax_ch = np.percentile(img, [2, 98])
    range_m = range_axes[label]
    n_pings = img.shape[1]

    im = ax.imshow(
        img,
        aspect='auto',
        cmap='viridis',
        vmin=vmin_ch, vmax=vmax_ch,
        origin='upper',
        extent=[0, n_pings, range_m[-1], 0],   # x: pings, y: metres (0 top)
    )
    ax.set_title(f"{label}\n{img.shape[0]} depth samples × {n_pings} pings", fontsize=12)
    ax.set_xlabel('Ping number')
    ax.set_ylabel('Range (m)')
    cbar = plt.colorbar(im, ax=ax, shrink=0.85)
    cbar.set_label('Power (dB)')

plt.suptitle(f"Echograms — {sample_file.name}", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Single-ping vertical profiles for both frequencies
PING_IDX = 500

fig, axes = plt.subplots(1, 2, figsize=(12, 7), sharey=False)

for ax, (label, img) in zip(axes, power_images.items()):
    ping_idx = min(PING_IDX, img.shape[1] - 1)
    profile = img[:, ping_idx]
    range_m = range_axes[label]
    ax.plot(profile, range_m, lw=1.5)
    ax.invert_yaxis()
    ax.set_xlabel('Power (dB)')
    ax.set_ylabel('Range (m)')
    ax.set_title(f'{label} — ping {ping_idx}')
    ax.grid(True, alpha=0.3)

plt.suptitle('Single-ping depth profiles', fontsize=12)
plt.tight_layout()
plt.show()